# Yolo runs

In [ ]:
import pandas as pd 
import numpy as np 

import seaborn as sns 
import matplotlib.pyplot as plt 

from pathlib import Path 

In [ ]:
FOLDER = Path('/home/leostre/Рабочий стол/SoftStairs-QAT/runs/detect')
baseline = 'train'

def read_data(run):
    return pd.read_csv(FOLDER / run / 'results.csv')

In [ ]:
from itertools import product 

def compare_runs(run, baseline):
    drop_cols = ['time']
    run_df = read_data(run)#.set_index(['epoch'])
    baseline_df = read_data(baseline)#.set_index(['epoch'])

    # equal_epochs = min(len(run_df), len(baseline_df))
    # run_df = run_df.iloc[:equal_epochs]
    # baseline_df = baseline_df.iloc[:equal_epochs]

    # plot losses 

    fig, axes = plt.subplots(2, 3, sharex=True, figsize=(12, 10))
    axes = np.array(axes).flatten()
    for ax, (mode, loss_name) in zip(axes, product(['train', 'val'], ['box_loss', 'cls_loss', 'l1_loss'])):
        sns.lineplot(run_df, x='epoch', y=f'{mode}/{loss_name}', ax=ax, label=run)
        sns.lineplot(baseline_df, x='epoch', y=f'{mode}/{loss_name}', ax=ax, label=baseline)
        ax.grid()
    
    plt.show()

    # plot metrics

    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(12, 10))
    axes = np.array(axes).flatten()
    for ax, (mode, loss_name) in zip(axes, product(['metrics'],  ['precision(B)', 'recall(B)', 'mAP50(B)', 'mAP50-95(B)'])):
        sns.lineplot(run_df, x='epoch', y=f'{mode}/{loss_name}', ax=ax, label=run)
        sns.lineplot(baseline_df, x='epoch', y=f'{mode}/{loss_name}', ax=ax, label=baseline)
        ax.grid()
    
    plt.show()



    

In [ ]:
compare_runs(
     'linear-0.1-4b-10e',
    # 'cos-0.01-4b-40e',
    'qat-Int8-10e',
#     'linear-0.5-4b-10e'
    )

# Activation Gathering

In [ ]:
from ultralytics import YOLO 

model = YOLO("yolo26n.pt")

In [ ]:
GATHERING_PARAMS = [
    'model.model.0.conv.weight',
    'model.model.23.one2one_cv3.2.1.1.conv.weight'
]

In [ ]:
import torch 
from dataclasses import dataclass
from functools import partial 
from pathlib import Path

@dataclass
class Controller:
    is_active: bool
    epoch: int

def add_grad_gathering(model:torch.nn.Module, parameters, controller, path):
    path = Path(path)
    collector = {}
    def save_grad(grad, parameter_name, path, controller):
            collector[parameter_name] = grad
            full_path = path / parameter_name 
            full_path.mkdir(exist_ok=True, parents=True)
            full_path = full_path/ f'{controller.epoch}.pt'
            if parameter_name in collector:
                torch.save(collector[parameter_name], full_path)
            

    for parameter_name in parameters:
        
        p = model.get_parameter(parameter_name)
        p.requires_grad = True
        p.register_post_accumulate_grad_hook(partial(save_grad, parameter_name=parameter_name, path=path, controller=controller))

def hook_yolo(model, path, parameters):
    controller = Controller(False, 0)
    add_grad_gathering(model, parameters, controller, path)
    def epoch_start(*args, **kwargs):
        controller.epoch += 1
        controller.is_active = True

    def on_train_batch_end(*args, **kwargs):
        controller.is_active = True
    model.add_callback('on_train_epoch_start', epoch_start)
    model.add_callback('on_train_batch_end', on_train_batch_end)


hook_yolo(model, '/home/leostre/Рабочий стол/SoftStairs-QAT/runs/grads', GATHERING_PARAMS)



In [ ]:
model.train(epochs=, imgsz=640)